In [2]:
import pandas as pd
import sqlite3

In [4]:
df = pd.read_excel(r"C:\Users\Rutvi\OneDrive\Desktop\Streaming Churn Intelligence Behavioral Segmentation & Retention Analytics\archive (6)\netflix_large_user_data.xlsx")

df.head()

,Customer ID,Subscription Length (Months),Customer Satisfaction Score (1-10),Daily Watch Time (Hours),Engagement Rate (1-10),Device Used Most Often,Genre Preference,Region,Payment History (On-Time/Delayed),Subscription Plan,Churn Status (Yes/No),Support Queries Logged,Age,Monthly Income ($),Promotional Offers Used,Number of Profiles Created
0,C00001,12,10,4.85,4,Tablet,Action,Europe,On-Time,Basic,No,10,33,6250,5,2
1,C00002,12,8,1.75,9,Laptop,Thriller,Europe,On-Time,Basic,Yes,9,28,7018,1,5
2,C00003,3,4,2.75,9,Smart TV,Comedy,Asia,On-Time,Premium,Yes,3,18,1055,1,5
3,C00004,3,7,3.00,9,Smart TV,Drama,Europe,Delayed,Premium,No,5,32,6707,5,4
4,C00005,24,2,1.37,5,Mobile,Drama,North America,On-Time,Standard,Yes,2,59,1506,3,5


In [5]:
conn = sqlite3.connect(":memory:")

df.to_sql("churn_data", conn, index=False, if_exists="replace")

1000

In [6]:
query = """
SELECT 
    "Churn Status (Yes/No)" AS churn_status,
    COUNT(*) AS total_customers
FROM churn_data
GROUP BY "Churn Status (Yes/No)"
"""

pd.read_sql(query, conn)

,churn_status,total_customers
0,No,461
1,Yes,539


In [7]:
query = """
SELECT 
    Region,
    "Churn Status (Yes/No)" AS churn_status,
    COUNT(*) AS total_customers
FROM churn_data
GROUP BY Region, "Churn Status (Yes/No)"
ORDER BY Region, total_customers DESC
"""

pd.read_sql(query, conn)

,Region,churn_status,total_customers
0,Africa,Yes,113
1,Africa,No,90
2,Asia,Yes,114
3,Asia,No,91
4,Europe,Yes,108
5,Europe,No,93
6,North America,Yes,103
7,North America,No,87
8,South America,Yes,101
9,South America,No,100


In [8]:
query = """
SELECT 
    "Subscription Plan",
    "Churn Status (Yes/No)" AS churn_status,
    COUNT(*) AS total_customers
FROM churn_data
GROUP BY "Subscription Plan", "Churn Status (Yes/No)"
ORDER BY "Subscription Plan", total_customers DESC
"""

pd.read_sql(query, conn)

,Subscription Plan,churn_status,total_customers
0,Basic,Yes,175
1,Basic,No,144
2,Premium,Yes,176
3,Premium,No,153
4,Standard,Yes,188
5,Standard,No,164


In [9]:
query = """
SELECT *
FROM churn_data
WHERE "Engagement Rate (1-10)" < 4
AND "Churn Status (Yes/No)" = 'Yes'
LIMIT 20
"""

pd.read_sql(query, conn)

,Customer ID,Subscription Length (Months),Customer Satisfaction Score (1-10),Daily Watch Time (Hours),Engagement Rate (1-10),Device Used Most Often,Genre Preference,Region,Payment History (On-Time/Delayed),Subscription Plan,Churn Status (Yes/No),Support Queries Logged,Age,Monthly Income ($),Promotional Offers Used,Number of Profiles Created
0,C00006,3,4,4.95,3,Tablet,Romance,Africa,Delayed,Basic,Yes,7,69,5581,4,3
1,C00008,6,5,3.54,2,Laptop,Romance,South America,On-Time,Standard,Yes,4,49,3630,5,3
2,C00009,6,9,3.79,1,Laptop,Drama,Africa,On-Time,Basic,Yes,6,27,4535,1,4
3,C00017,6,2,0.78,2,Laptop,Drama,Asia,Delayed,Basic,Yes,10,43,9913,3,1
4,C00021,6,3,4.91,3,Tablet,Action,Asia,On-Time,Premium,Yes,3,70,5614,0,4
5,C00023,24,7,0.81,1,Desktop,Documentary,North America,On-Time,Premium,Yes,7,67,6720,3,5
6,C00031,12,8,1.41,1,Desktop,Action,Africa,On-Time,Premium,Yes,7,20,6097,1,5
7,C00032,12,6,3.77,2,Tablet,Sci-Fi,Africa,Delayed,Premium,Yes,8,56,3059,1,2
8,C00035,3,10,1.68,1,Desktop,Sci-Fi,Europe,Delayed,Standard,Yes,3,37,7751,0,1
9,C00047,24,4,3.50,3,Desktop,Documentary,Africa,On-Time,Premium,Yes,10,56,6064,0,4


In [10]:
query = """
SELECT 
    COUNT(*) AS total_customers,
    
    SUM(CASE WHEN "Churn Status (Yes/No)" = 'Yes' THEN 1 ELSE 0 END) AS total_churned,
    
    ROUND(
        1.0 * SUM(CASE WHEN "Churn Status (Yes/No)" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS churn_rate,

    ROUND(AVG("Engagement Rate (1-10)"), 2) AS avg_engagement,
    
    ROUND(AVG("Customer Satisfaction Score (1-10)"), 2) AS avg_satisfaction

FROM churn_data
"""

pd.read_sql(query, conn)

,total_customers,total_churned,churn_rate,avg_engagement,avg_satisfaction
0,1000,539,0.539,5.51,5.38


# 📓 Notebook 09 — SQL Layer (Final Insights)

## 📊 Key Insights from SQL Analysis

From the SQL-based analysis of the churn dataset, we derived several important business insights that help understand customer behavior and churn patterns.

---

## 📉 1. Churn Distribution
- The dataset shows a **significant number of customers have churned**
- This indicates churn is a **core business problem**, not an outlier issue
- Both churned and retained users are meaningfully represented

---

## 🌍 2. Region-wise Churn Behavior
- Churn is **unevenly distributed across regions**
- Some regions show higher churn concentration than others
- This suggests:
  - regional preferences differ
  - service/content performance may vary by location
  - targeted regional strategies are needed

---

## 💳 3. Subscription Plan Impact
- Churn varies across different subscription plans
- Certain plans show higher churn rates compared to others
- Possible reasons:
  - pricing sensitivity
  - perceived value differences
  - feature limitations in lower tiers

---

## ⚠️ 4. High-Risk Customers
- Users with **low engagement (< 4)** who have churned represent a high-risk behavior pattern
- These users typically show:
  - early drop-off behavior
  - low platform interaction
  - low retention probability

---

## 📊 5. Overall KPI Snapshot
- The dataset shows a **noticeable churn rate across the user base**
- Engagement and satisfaction are key indicators of retention
- Higher engagement generally correlates with lower churn risk

---

## 💡 Business Impact

This SQL layer demonstrates how structured queries enable:

- Fast identification of churn patterns
- Clear segmentation for retention strategies
- Data-driven business decision-making
- Scalable reporting for dashboards and BI tools

---

## 🚀 Final Takeaway

> SQL analysis confirms that churn is strongly influenced by engagement behavior, subscription type, and regional differences. These insights can directly guide targeted retention strategies and improve customer lifetime value.
